# Lab: Context 전략 실습 과제 - 정답 코드

**도메인**: 기술 지원 티켓 분류

3가지 컨텍스트 전략(System 없음 / 역할+제약 / Few-shot)을 PydanticAI Agent로 적용하고,
토큰/비용/정확도를 비교한다.

> **Jupyter 노트북 실행 참고**: `agent.run_sync()` 대신 `await agent.run()`을 사용한다.
> Jupyter는 이미 이벤트 루프가 실행 중이므로 `run_sync()`는 `RuntimeError`를 일으킨다.

## 환경 설정

필요한 라이브러리를 임포트하고 `.env`에서 API 키를 로드한다.

In [ ]:
import time
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from enum import Enum
from dotenv import load_dotenv

# .env 파일에서 OPENAI_API_KEY 등 환경변수를 로드
load_dotenv()

## 모델 및 비용 설정

비용 계산 함수를 정의한다. 입력/출력 토큰 단가가 다르므로 분리하여 계산한다.

In [ ]:
# 사용할 모델 지정
MODEL = "openai:gpt-5.4"

# 비용 단가 ($/1M tokens) — 출력 토큰이 입력보다 약 4배 비싸다
PRICING = {
    "openai:gpt-5.4": {"input": 2.50, "output": 10.00},
    "openai:gpt-5.4-mini": {"input": 0.15, "output": 0.60},
}


def calculate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """토큰 수와 단가로 비용(USD)을 계산한다"""
    prices = PRICING.get(model, {"input": 2.50, "output": 10.00})
    return (input_tokens * prices["input"] + output_tokens * prices["output"]) / 1_000_000

---
## Pydantic 스키마: 기술 지원 티켓 분류

`Enum`으로 카테고리와 우선순위를 제한한다.
LLM이 범위 밖의 값을 생성하는 것을 원천 차단.

| 타입 | 역할 |
|------|------|
| `Category` (Enum) | 4가지 카테고리로 제한 |
| `Priority` (Enum) | 3단계 우선순위로 제한 |
| `TicketAnalysis` (BaseModel) | 응답 스키마 — `output_type`으로 전달 |

In [ ]:
# Enum으로 카테고리를 4가지로 제한 — 범위 밖 값 생성 원천 차단
class Category(str, Enum):
    """기술 지원 카테고리"""
    BUG = "버그"               # 기존 기능의 오작동, 에러, 장애
    FEATURE = "기능요청"        # 새 기능 추가 또는 기존 기능 개선 요청
    QUESTION = "사용법문의"     # 문서, 설정, API 사용법 질문
    INFRA = "인프라"            # 서버, DB, 네트워크, 성능 관련


# Enum으로 우선순위를 3단계로 제한
class Priority(str, Enum):
    """우선순위"""
    HIGH = "높음"              # 서비스 중단, 데이터 손실, 보안 취약점
    MEDIUM = "보통"            # 기능 불편, 간헐적 오류, 우회 가능
    LOW = "낮음"               # 단순 질문, 기능 제안, 문서 요청


# Pydantic 모델 — output_type으로 전달하면 LLM 응답이 이 스키마에 강제된다
class TicketAnalysis(BaseModel):
    """기술 지원 티켓 분석 결과

    PydanticAI의 output_type으로 사용하면
    LLM 응답이 이 스키마에 맞게 자동으로 강제된다.
    """
    category: Category = Field(
        description="티켓 카테고리. 장애/오류면 '버그', 새 기능 요청이면 '기능요청', "
                    "사용법 질문이면 '사용법문의', 서버/DB/네트워크면 '인프라'"
    )
    priority: Priority = Field(
        description="우선순위. 서비스 중단/데이터 손실이면 '높음', "
                    "기능 불편이면 '보통', 단순 질문/개선 제안이면 '낮음'"
    )
    summary: str = Field(
        description="티켓 핵심을 1문장으로 요약"
    )

---
## 테스트 데이터

쉬운 케이스부터 경계 케이스까지 다양하게 구성한다.
경계 케이스가 포함되어야 전략 간 차이가 명확하게 드러난다.

각 튜플: `(입력 텍스트, 기대 카테고리, 기대 우선순위)`

In [ ]:
# 5개 테스트 케이스 — 쉬운 것부터 경계 케이스까지
TEST_CASES = [
    (   # 쉬운 케이스: 명확한 인프라 장애 + 높은 우선순위
        "프로덕션 DB 커넥션 풀이 가득 차서 API가 500 에러를 반환합니다. 긴급 조치 필요합니다.",
        "인프라", "높음",
    ),
    (   # 쉬운 케이스: 명확한 기능요청 + 낮은 우선순위
        "대시보드에서 CSV 내보내기 기능 추가해주세요. 매번 수동 복붙하고 있어요.",
        "기능요청", "낮음",
    ),
    (   # 경계 케이스: 버그이지만 우회 가능 → 보통 우선순위
        "로그인할 때 가끔 비밀번호를 맞게 입력해도 실패해요. 재시도하면 됩니다.",
        "버그", "보통",
    ),
    (   # 쉬운 케이스: 단순 질문
        "API 문서에서 rate limit 관련 설명이 어디 있나요?",
        "사용법문의", "낮음",
    ),
    (   # 경계 케이스: 인프라 + 데이터 유실 → 높은 우선순위
        "배치 작업이 새벽 3시마다 메모리 부족으로 중단됩니다. 데이터가 유실되고 있어요.",
        "인프라", "높음",
    ),
]

---
## 전략 A: System Prompt 없이 호출

`instructions`를 지정하지 않으면 LLM에게 분류 기준이 주어지지 않는다.
`output_type` 덕분에 출력 스키마는 강제되지만, 분류 판단은 LLM의 일반 지식에만 의존한다.

토큰이 가장 적지만 경계 케이스에서 판단이 흔들릴 수 있다.

In [ ]:
async def strategy_no_system(text: str) -> tuple[TicketAnalysis, dict]:
    """System Prompt(instructions) 없이 User 메시지만으로 분류를 요청한다"""
    start = time.time()

    # instructions 없이 Agent 생성 — output_type만 지정
    agent = Agent(MODEL, output_type=TicketAnalysis)
    # Jupyter에서는 await agent.run() 사용 (run_sync 대신)
    result = await agent.run(f"다음 기술 지원 티켓을 분류하라:\n\n{text}")

    elapsed = time.time() - start
    usage = result.usage()
    meta = {
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "latency": elapsed,
        "cost": calculate_cost(MODEL, usage.input_tokens, usage.output_tokens),
    }
    return result.output, meta

---
## 전략 B: 역할 + 제약이 있는 System Prompt

`instructions`에 역할, 카테고리 기준, 우선순위 기준을 명시한다.
분류 판단의 근거를 LLM에게 제공하여 정확도를 높인다.

토큰 증가는 소폭이지만 비용 대비 효과가 매우 크다.
대부분의 실무 상황에서 가장 효율적인 전략이다.

In [ ]:
async def strategy_with_constraints(text: str) -> tuple[TicketAnalysis, dict]:
    """역할 지정 + 분류 기준이 포함된 instructions로 호출한다"""
    start = time.time()

    # instructions에 역할 + 카테고리/우선순위 기준을 명시
    agent = Agent(
        MODEL,
        instructions=(
            "당신은 SaaS 기업의 기술 지원 티켓 분류 전문가입니다.\n\n"
            "카테고리 기준:\n"
            "- 버그: 기존 기능의 오작동, 에러, 장애\n"
            "- 기능요청: 새 기능 추가 또는 기존 기능 개선 요청\n"
            "- 사용법문의: 문서, 설정, API 사용법 질문\n"
            "- 인프라: 서버, DB, 네트워크, 성능, 배치 작업 관련\n\n"
            "우선순위 기준:\n"
            "- 높음: 서비스 중단, 데이터 손실/유실, 보안 취약점\n"
            "- 보통: 기능 불편, 간헐적 오류, 우회 가능한 문제\n"
            "- 낮음: 단순 질문, 기능 제안, 문서 요청"
        ),
        output_type=TicketAnalysis,
    )
    # Jupyter에서는 await agent.run() 사용
    result = await agent.run(text)

    elapsed = time.time() - start
    usage = result.usage()
    meta = {
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "latency": elapsed,
        "cost": calculate_cost(MODEL, usage.input_tokens, usage.output_tokens),
    }
    return result.output, meta

---
## 전략 C: Few-shot 예시 포함

`instructions`에 경계 케이스 예시를 포함한다.
모호한 입력에서의 분류 정확도가 향상된다.
단, 토큰 사용량이 가장 크므로 비용/정확도 트레이드오프를 고려한다.

예시에 **분류 이유**를 함께 제공하면 LLM의 추론 품질이 더 높아진다.

In [ ]:
async def strategy_few_shot(text: str) -> tuple[TicketAnalysis, dict]:
    """Few-shot 예시를 instructions에 포함하여 호출한다"""
    start = time.time()

    # instructions에 역할 + 기준 + 경계 케이스 예시(분류 이유 포함)
    agent = Agent(
        MODEL,
        instructions=(
            "당신은 SaaS 기업의 기술 지원 티켓 분류 전문가입니다.\n\n"
            "카테고리: 버그, 기능요청, 사용법문의, 인프라\n"
            "우선순위: 높음, 보통, 낮음\n\n"
            "분류 예시:\n\n"
            "예시 1 (경계 케이스 - 간헐적 오류):\n"
            "입력: '검색 기능이 가끔 결과를 안 보여줘요. 새로고침하면 나옵니다.'\n"
            "분류: 카테고리=버그, 우선순위=보통, 요약='검색 결과 간헐적 미표시'\n"
            "이유: 오작동이므로 '버그', 우회 가능하므로 '보통'\n\n"
            "예시 2 (경계 케이스 - 인프라 vs 사용법문의):\n"
            "입력: '디스크 사용량이 90% 넘었는데 알림이 안 왔어요. 설정 방법 알려주세요.'\n"
            "분류: 카테고리=사용법문의, 우선순위=낮음, 요약='디스크 알림 설정 방법 문의'\n"
            "이유: 장애가 아닌 설정 질문이므로 '사용법문의'"
        ),
        output_type=TicketAnalysis,
    )
    # Jupyter에서는 await agent.run() 사용
    result = await agent.run(text)

    elapsed = time.time() - start
    usage = result.usage()
    meta = {
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "latency": elapsed,
        "cost": calculate_cost(MODEL, usage.input_tokens, usage.output_tokens),
    }
    return result.output, meta

---
## 비교 실행

3가지 전략을 동일한 5개 테스트 케이스에 적용하고 정확도/비용/지연을 비교한다.
전략 함수들이 `async def`이므로 비교 함수도 `async`로 정의한다.

In [ ]:
async def run_comparison():
    """3가지 전략을 동일한 테스트 케이스에 적용하고 비교 리포트를 출력한다"""
    strategies = [
        ("A. System 없음", strategy_no_system),
        ("B. 역할 + 제약", strategy_with_constraints),
        ("C. Few-shot", strategy_few_shot),
    ]

    print("Context 전략 비교 결과 (정답)")
    print("=" * 70)

    summary_rows = []

    for strategy_name, strategy_fn in strategies:
        print(f"\n[{strategy_name}]")
        print(f"{'─'*70}")

        correct = 0
        total_cost = 0.0
        total_latency = 0.0
        total_input_tokens = 0
        total_output_tokens = 0

        for i, (text, expected_cat, expected_pri) in enumerate(TEST_CASES, 1):
            try:
                # 전략 함수가 async이므로 await로 호출
                result, meta = await strategy_fn(text)

                cat_ok = result.category.value == expected_cat
                pri_ok = result.priority.value == expected_pri
                is_correct = cat_ok and pri_ok
                if is_correct:
                    correct += 1

                total_cost += meta["cost"]
                total_latency += meta["latency"]
                total_input_tokens += meta["input_tokens"]
                total_output_tokens += meta["output_tokens"]

                mark = "O" if is_correct else "X"
                print(
                    f"  {i}. 예측: {result.category.value}/{result.priority.value}, "
                    f"정답: {expected_cat}/{expected_pri} -> {mark}"
                )
            except Exception as e:
                print(f"  {i}. [오류] {e}")

        accuracy = correct / len(TEST_CASES) * 100
        print(f"\n  정확도:     {correct}/{len(TEST_CASES)} ({accuracy:.0f}%)")
        print(f"  총 비용:     ${total_cost:.6f}")
        print(f"  총 지연:     {total_latency:.2f}초")
        print(f"  입력 토큰:   {total_input_tokens}")
        print(f"  출력 토큰:   {total_output_tokens}")

        summary_rows.append({
            "strategy": strategy_name,
            "accuracy": accuracy,
            "cost": total_cost,
            "latency": total_latency,
            "input_tokens": total_input_tokens,
            "output_tokens": total_output_tokens,
        })

    # 최종 비교 테이블
    print(f"\n{'전략':<20} {'정확도':>8} {'비용($)':>10} {'지연(초)':>8} {'입력토큰':>8} {'출력토큰':>8}")
    print(f"{'─'*75}")
    for row in summary_rows:
        print(
            f"{row['strategy']:<20} {row['accuracy']:>7.0f}% "
            f"{row['cost']:>10.6f} {row['latency']:>8.2f} "
            f"{row['input_tokens']:>8} {row['output_tokens']:>8}"
        )


# async 함수이므로 await로 호출
await run_comparison()

---
## 정답 해설

**1. System Prompt 없음 (전략 A)**
- 토큰이 가장 적지만 분류 기준이 불명확하여 정확도가 낮을 수 있다
- 특히 경계 케이스(간헐적 오류, 인프라 vs 버그)에서 판단이 흔들린다
- `output_type` 덕분에 '형식'은 보장되지만 '내용'의 정확도는 별개

**2. 역할 + 제약 (전략 B)**
- `instructions`에 분류 기준을 명시하면 정확도가 크게 향상된다
- 토큰 증가는 소폭이지만, 비용 대비 효과가 매우 크다
- 대부분의 실무 상황에서 가장 효율적인 전략

**3. Few-shot (전략 C)**
- 경계 케이스에 대한 예시를 제공하면 모호한 입력에서의 정확도가 향상된다
- 토큰 증가가 가장 크므로, 모든 입력에 적용하기보다 어려운 케이스에 선택적으로 사용하는 것이 비용 효율적

**핵심 교훈:**
- 단순 작업: `instructions` + 제약만으로 충분 (전략 B)
- 경계 케이스가 많은 작업: Few-shot 추가 (전략 C)
- Agent 파이프라인: 단계마다 적절한 전략을 선택하라
- PydanticAI `output_type`: '형식'을 보장하고, `instructions`: '내용'의 정확도를 높인다

LangSmith 대시보드에서 각 전략의 트레이스를 비교하세요!
https://smith.langchain.com